# OTC Medicine Information Organization and Retrieval System

## Data Preparation and Taxonomy Development

The collection consists of 300 over-the-counter medicines
across five therapeutic groups.

The objective of this notebook is to:
- Clean and standardize existing metadata.
- Prepare the collection for faceted classification.
- Develop a reproducible stratified sample for manual classification.
- Prepare ground truth data for automatic classification.

In [1]:
import pandas as pd
import numpy as np

filename = "/content/Final Info 202 dataset 2.csv"

df = pd.read_csv(filename, header=1)

print("Dataset dimensions:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

display(df.head())

Dataset dimensions: (304, 13)

Column names:
['Unnamed: 0', 'Groups', 'Item', 'Product', 'Intended Use', 'Intended User', 'Dosage Form', 'Brand Type', 'Active Ingredient', 'Uses', 'Selected Warnings', 'Selected texts (Uses + Warnings)', 'Initial Codes']


,Unnamed: 0,Groups,Item,Product,Intended Use,Intended User,Dosage Form,Brand Type,Active Ingredient,Uses,Selected Warnings,Selected texts (Uses + Warnings),Initial Codes
0,NaN,Pain and Fever,1.0,Tylenol Extra Strength,Pain; Fever,Adult,Caplet,National brand,Acetaminophen 500 mg,"Relieves minor aches/pains from cold, headache...",Severe liver damage possible with excessive us...,"Relieves headaches, backaches, muscular aches,...",relieving multiple types of pain; reducing fev...
1,NaN,Pain and Fever,2.0,Advil,Pain; Fever,Adult,Tablet/Caplet,National brand,Ibuprofen 200 mg,"Relieves headache, toothache, backache, menstr...",NSAID may cause severe stomach bleeding; may i...,"Relieves minor aches, pains and fever. NSAIDs ...",relieving everyday pain; reducing fever; warni...
2,NaN,Pain and Fever,3.0,Aleve Back & Muscle Pain,Pain; Fever,Adult,Caplet,National brand,Naproxen sodium 220 mg,"Relieves backache, muscular aches, arthritis p...",NSAID may cause severe stomach bleeding; napro...,"Relieves backache, muscular aches, arthritis p...",targeting muscle and back pain; relieving othe...
3,NaN,Pain and Fever,4.0,Bayer Genuine Aspirin,Pain; Fever,Adult*,Tablet,National brand,Aspirin 325 mg,"Relieves headache, muscle pain, toothache, men...",Children/teens recovering from chickenpox or f...,"Relieves headache, muscle pain, toothache, men...",relieving multiple types of pain; reducing fev...
4,NaN,Pain and Fever,5.0,Children's Tylenol,Pain; Fever,Child,Chewable tablet,National brand,Acetaminophen 160 mg,"Relieves pain from cold, flu, headache, sore t...",Severe liver damage possible if child exceeds ...,"Relieves childhood aches, sore throat, toothac...",relieving childhood pain; reducing fever; limi...


## Data Cleaning

Remove non-product rows and unnecessary columns,
standardize text formatting, and check for missing
values and duplicate item IDs.

The original dataset is preserved for reproducibility.

In [2]:
# Create a separate copy of the original dataset
df_clean = df.copy()

# Fill down therapeutic group names
df_clean["Groups"] = df_clean["Groups"].ffill()

# Remove rows without an item ID or product name
df_clean = df_clean.dropna(subset=["Item", "Product"])

# Remove unnecessary unnamed columns
df_clean = df_clean.loc[:, ~df_clean.columns.str.startswith("Unnamed")]

# Convert item IDs to integers
df_clean["Item"] = df_clean["Item"].astype(int)

# Remove unnecessary spaces from text
for col in df_clean.select_dtypes(include="object").columns:
    df_clean[col] = df_clean[col].str.strip()

# Reset row index
df_clean = df_clean.reset_index(drop=True)

# Check cleaned dataset
print("Cleaned dataset dimensions:", df_clean.shape)

print("\nMedicines per therapeutic group:")
print(df_clean["Groups"].value_counts())

print("\nMissing values:")
print(df_clean.isnull().sum())

print("\nDuplicate item IDs:")
print(df_clean["Item"].duplicated().sum())

Cleaned dataset dimensions: (300, 12)

Medicines per therapeutic group:
Groups
Pain and Fever           60
Cough Cold & Flu         60
Allergy & Respiratory    60
Digestive & Heartburn    60
Skin & Topical           60
Name: count, dtype: int64

Missing values:
Groups                              0
Item                                0
Product                             0
Intended Use                        0
Intended User                       0
Dosage Form                         0
Brand Type                          0
Active Ingredient                   0
Uses                                0
Selected Warnings                   0
Selected texts (Uses + Warnings)    0
Initial Codes                       0
dtype: int64

Duplicate item IDs:
0


## Inspection of Existing Facets

Examine the unique categories and their frequencies
within the four existing facets:

- Intended Use
- Intended User
- Dosage Form
- Brand Type

This step helps identify inconsistent terminology,
overlapping categories, and opportunities for
hierarchical classification.

In [3]:
# Define existing facets
facets = [
    "Intended Use",
    "Intended User",
    "Dosage Form",
    "Brand Type"
]

# Examine categories within each facet
for facet in facets:
    print("=" * 60)
    print("FACET:", facet)
    print("Number of unique categories:", df_clean[facet].nunique())
    print("\nCategory distribution:")
    print(df_clean[facet].value_counts().to_string())
    print()

FACET: Intended Use
Number of unique categories: 47

Category distribution:
Intended Use
Pain; Fever                                                       56
Digestive; Heartburn                                              40
Allergy; Respiratory                                              40
Skin; Topical                                                     40
Cold; Flu; Cough                                                  20
Cough Cold; Flu                                                   20
Allergy                                                           11
Pain; Muscle                                                       4
Heartburn; Acid indigestion; Sour stomach                          4
Skin; Itching                                                      4
Heartburn                                                          4
Cold; Flu; Cough; Fever; Pain; Congestion                          4
Allergy; Congestion                                                3
Cough; Throat 

## Standardization of Intended Use

The original Intended Use field contains combinations
of therapeutic purposes and symptoms.

To support multi-label faceted browsing, these
combinations are separated into individual tags.

The original field is retained to preserve the
source information.

In [4]:
# Define a function to standardize intended use

def standardize_use(value):

    # Separate individual uses
    tags = value.split(";")

    # Remove extra spaces and standardize capitalization
    tags = [tag.strip().lower() for tag in tags]

    # Standardize inconsistent terminology
    mapping = {
        "cough cold": ["cough", "cold"],
        "pain relief": ["pain"],
        "runny nose/sneezing": ["runny nose", "sneezing"]
    }

    standardized_tags = []

    for tag in tags:
        standardized_tags.extend(mapping.get(tag, [tag]))

    # Remove empty and duplicate tags
    return sorted(set(tag for tag in standardized_tags if tag))


# Create a new column without changing the original
df_clean["Intended Use Tags"] = (
    df_clean["Intended Use"].apply(standardize_use)
)

# Count individual intended use tags
use_counts = (
    df_clean["Intended Use Tags"]
    .explode()
    .value_counts()
)

print("Number of individual use tags:", len(use_counts))

print("\nFrequency of individual use tags:")
print(use_counts.to_string())

print("\nExamples of standardized data:")
display(
    df_clean[
        ["Product", "Intended Use", "Intended Use Tags"]
    ].head(10)
)

Number of individual use tags: 50

Frequency of individual use tags:
Intended Use Tags
pain                    68
fever                   63
allergy                 60
cough                   57
heartburn               52
cold                    49
flu                     48
skin                    47
respiratory             40
topical                 40
digestive               40
congestion              12
acid indigestion         7
itching                  7
sour stomach             6
diarrhea                 4
throat                   4
muscle                   4
scrapes                  3
minor cuts               3
scalp                    3
dandruff                 3
infection prevention     3
burns                    3
runny nose               3
constipation             3
rash                     3
gas                      3
upset stomach            3
fungal                   2
sneezing                 2
chest congestion         2
nasal symptoms           2
eyes                  

,Product,Intended Use,Intended Use Tags
0,Tylenol Extra Strength,Pain; Fever,"[fever, pain]"
1,Advil,Pain; Fever,"[fever, pain]"
2,Aleve Back & Muscle Pain,Pain; Fever,"[fever, pain]"
3,Bayer Genuine Aspirin,Pain; Fever,"[fever, pain]"
4,Children's Tylenol,Pain; Fever,"[fever, pain]"
5,Children's Motrin,Pain; Fever,"[fever, pain]"
6,Children's Motrin,Pain; Fever,"[fever, pain]"
7,CVS Acetaminophen,Pain; Fever,"[fever, pain]"
8,Walgreens Ibuprofen 200,Pain; Fever,"[fever, pain]"
9,Equate Pain Reliever,Pain; Fever,"[fever, pain]"


## Standardization of Intended User

The Intended User facet represents the target
population of each OTC medicine.

Formatting inconsistencies are standardized while
preserving age-specific restrictions and qualifications.

The original metadata is retained to avoid losing
potentially important safety information.

In [5]:
# Define mapping for inconsistent user categories

user_mapping = {
    "Adult/12+": "Adult / 12+",
    "Adult; Child": "Adult / Child"
}

# Create a standardized column
df_clean["Intended User Standardized"] = (
    df_clean["Intended User"].replace(user_mapping)
)

# Examine standardized categories
print("Original number of categories:",
      df_clean["Intended User"].nunique())

print("Standardized number of categories:",
      df_clean["Intended User Standardized"].nunique())

print("\nStandardized category distribution:")
print(
    df_clean["Intended User Standardized"]
    .value_counts()
    .to_string()
)

# Check for missing values
print("\nMissing values:",
      df_clean["Intended User Standardized"].isnull().sum())

Original number of categories: 13
Standardized number of categories: 11

Standardized category distribution:
Intended User Standardized
Adult                153
Adult / 12+           86
Child                 32
Adult / Child          8
Adult / Child 6+       7
Adult / Child 2+       6
Adult / Child 12+      3
Child (2–11)           2
Adult*                 1
Adult / 17+            1
Adult / Child 2+*      1

Missing values: 0


## Hierarchical Classification of Dosage Form

The original dataset contains 52 distinct dosage forms.

To reduce browsing complexity, a two-level hierarchy
is introduced:

- Level 1: Broad dosage form family
- Level 2: Specific dosage form

The original dosage form is preserved to retain
meaningful distinctions between formulations.

Each dosage form is assigned to one broad family
to support mutually exclusive parent categories.

In [6]:
# Define broad dosage form families

dosage_hierarchy = {

    "Tablets and Caplets": [
        "Tablet", "Caplet", "Chewable tablet",
        "Extended-release tablet",
        "Extended-release caplet",
        "Orally disintegrating tablet",
        "Delayed-release tablet",
        "Effervescent tablet",
        "Enteric-coated tablet",
        "Film-coated tablet", "Tablet/Caplet"
    ],

    "Capsules": [
        "Capsule", "Softgel", "Gelcap",
        "Liquid-filled capsule",
        "Delayed-release capsule"
    ],

    "Liquids": [
        "Liquid", "Oral suspension",
        "Oral solution", "Oral liquid", "Syrup"
    ],

    "Powders and Granules": [
        "Powder", "Powder for hot liquid",
        "Powder for oral solution",
        "Powder packets", "Oral granules"
    ],

    "Creams and Semisolids": [
        "Cream", "Ointment", "Gel",
        "Lotion", "Paste", "Jelly",
        "Vaginal cream"
    ],

    "Sprays and Rinses": [
        "Nasal spray", "Throat spray",
        "Topical spray", "Spray", "Nasal rinse"
    ],

    "Drops": [
        "Eye drops", "Oral drops"
    ],

    "Lozenges and Chews": [
        "Lozenge", "Chew"
    ],

    "External Applications": [
        "Patch", "Heat wrap", "Roll-on",
        "Topical applicator", "Medicated pad"
    ],

    "Washes and Cleansers": [
        "Shampoo", "Wash", "Cleanser"
    ],

    "Rectal Preparations": [
        "Rectal suppository", "Rectal enema"
    ]
}

# Create a mapping from specific form to broad family

dosage_mapping = {
    form: family
    for family, forms in dosage_hierarchy.items()
    for form in forms
}

# Assign broad dosage form categories

df_clean["Dosage Form Family"] = (
    df_clean["Dosage Form"]
    .map(dosage_mapping)
    .fillna("Other")
)

# Check results

print("Original dosage forms:",
      df_clean["Dosage Form"].nunique())

print("Broad dosage form families:",
      df_clean["Dosage Form Family"].nunique())

print("\nDosage form family distribution:")
print(
    df_clean["Dosage Form Family"]
    .value_counts()
    .to_string()
)

print("\nUnmapped dosage forms:")
print(
    df_clean.loc[
        df_clean["Dosage Form Family"] == "Other",
        "Dosage Form"
    ].unique()
)

print("\nExamples of hierarchical classification:")

display(
    df_clean[
        ["Product", "Dosage Form Family", "Dosage Form"]
    ].head(15)
)

Original dosage forms: 52
Broad dosage form families: 11

Dosage form family distribution:
Dosage Form Family
Tablets and Caplets      102
Creams and Semisolids     53
Liquids                   44
Capsules                  23
Sprays and Rinses         22
Powders and Granules      17
Drops                     13
Washes and Cleansers       9
Lozenges and Chews         8
External Applications      7
Rectal Preparations        2

Unmapped dosage forms:
[]

Examples of hierarchical classification:


,Product,Dosage Form Family,Dosage Form
0,Tylenol Extra Strength,Tablets and Caplets,Caplet
1,Advil,Tablets and Caplets,Tablet/Caplet
2,Aleve Back & Muscle Pain,Tablets and Caplets,Caplet
3,Bayer Genuine Aspirin,Tablets and Caplets,Tablet
4,Children's Tylenol,Tablets and Caplets,Chewable tablet
5,Children's Motrin,Liquids,Oral suspension
6,Children's Motrin,Tablets and Caplets,Chewable tablet
7,CVS Acetaminophen,Tablets and Caplets,Caplet
8,Walgreens Ibuprofen 200,Tablets and Caplets,Caplet
9,Equate Pain Reliever,Tablets and Caplets,Caplet


## Brand Type Classification

The Brand Type facet classifies medicines according
to their commercial brand identity.

Four categories are retained:
- National brand
- Store brand
- Generic
- Unknown

The Unknown category accommodates products whose
brand identity cannot be confidently established.

The categories are mutually exclusive under the
current classification rules. However, the distinction
between generic and branded products can be ambiguous.

In [7]:
# Review brand type categories

print("Number of brand categories:",
      df_clean["Brand Type"].nunique())

print("\nBrand type distribution:")
print(df_clean["Brand Type"].value_counts())

print("\nPercentage distribution:")
print(
    df_clean["Brand Type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nProducts with unknown brand type:")

display(
    df_clean.loc[
        df_clean["Brand Type"] == "Unknown",
        ["Item", "Product", "Brand Type"]
    ]
)

Number of brand categories: 4

Brand type distribution:
Brand Type
National brand    270
Store brand        27
Generic             2
Unknown             1
Name: count, dtype: int64

Percentage distribution:
Brand Type
National brand    90.00
Store brand        9.00
Generic            0.67
Unknown            0.33
Name: proportion, dtype: float64

Products with unknown brand type:


,Item,Product,Brand Type
169,170,GoodSense Allergy Relief Loratadine 10 mg,Unknown


## Export Standardized Dataset

The standardized dataset preserves the original
metadata and incorporates three additional fields:

- Intended Use Tags
- Intended User Standardized
- Dosage Form Family

This dataset will be used for manual classification,
machine learning, faceted browsing, and retrieval.

In [8]:
# Create an export copy
df_export = df_clean.copy()

# Convert lists into semicolon-separated text
df_export["Intended Use Tags"] = (
    df_export["Intended Use Tags"]
    .apply(lambda tags: "; ".join(tags))
)

# Save standardized dataset
df_export.to_csv(
    "OTC_Medicines_Standardized.csv",
    index=False
)

print("Dataset saved successfully!")
print("Final dimensions:", df_export.shape)

print("\nFinal columns:")
print(df_export.columns.tolist())

Dataset saved successfully!
Final dimensions: (300, 15)

Final columns:
['Groups', 'Item', 'Product', 'Intended Use', 'Intended User', 'Dosage Form', 'Brand Type', 'Active Ingredient', 'Uses', 'Selected Warnings', 'Selected texts (Uses + Warnings)', 'Initial Codes', 'Intended Use Tags', 'Intended User Standardized', 'Dosage Form Family']


In [9]:
from google.colab import files

files.download("OTC_Medicines_Standardized.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>